## התאמת צירים: broadcasting והוספת ציר עם None (כללים 3 ו-4)

בסעיף 7.6 ראינו broadcasting בין מערך למספר בודד. עכשיו מכללים את זה לכלל המלא — **כלל 3** מארבעת הכללים:

```{admonition} כלל 3 — Broadcasting
:class: tip
שני מערכים מתחברים אם, **מימין לשמאל**, כל זוג צירים מתאימים או **שווה בגודלו**, או **שאחד מהם 1**.
```

הכלל הזה מאפשר לבנות בדיוק את ‎`Y`‎ שבנינו בלולאה בסעיף 9.1 — בלי לולאה בכלל. בהמשך הסעיף, **כלל 4** מראה איך מוסיפים למערך ציר בגודל 1 עם ‎`None`‎, כדי שכלל 3 יתקיים.

In [ ]:
import numpy as np

g = 9.8
theta = np.radians(45)
v0_nom = 20.0                                       # m/s
v0_runs = np.array([18.0, 19.5, 20.0, 20.5, 22.0])  # shape (5,)
n_runs, n_t = 5, 60
t_flight_nom = 2 * v0_nom * np.sin(theta) / g
t = np.linspace(0, t_flight_nom, n_t)               # shape (60,)

### דוגמה: בניית ‎`Y`‎ בפעולה וקטורית אחת

‎`v0_runs`‎ בצורה ‎`(5,)`‎ ו-‎`t`‎ בצורה ‎`(60,)`‎ — צורות שונות, אי אפשר לחבר ישירות. אבל אם מוסיפים ל-‎`v0_runs`‎ ציר בגודל 1 (‎`v0_runs[:, None]`‎, צורה ‎`(5, 1)`‎), הכלל מתקיים: מימין לשמאל, ‎`60`‎ מול ‎`1`‎ (אחד מהם 1 ✓), ו-‎`5`‎ מול ‎`1`‎ הנסתר של ‎`t`‎ (אחד מהם 1 ✓).

In [ ]:
Y = v0_runs[:, None] * np.sin(theta) * t - 0.5 * g * t**2
print("Y.shape:", Y.shape)   # (5, 60), without a loop

# check against the loop of 9.1:
Y_loop = np.zeros((n_runs, n_t))
for i in range(n_runs):
    Y_loop[i] = v0_runs[i] * np.sin(theta) * t - 0.5 * g * t**2

print(np.all(Y == Y_loop))

### באג נפוץ: לשכוח להוסיף ציר עם None

בלי ‎`[:, None]`‎, ‎`v0_runs`‎ נשאר בצורה ‎`(5,)`‎ — ו-NumPy מנסה להתאים אותו ל-‎`t`‎ בצורה ‎`(60,)`‎ ישירות: ‎`5`‎ מול ‎`60`‎, שניהם שונים מ-1. זה **לא** באג שקט: התא הבא נעצר ב-‎`ValueError`‎. זה דווקא נוח, כי הודעת השגיאה מציגה את שתי הצורות שלא מתאימות.

In [ ]:
bad = v0_runs * np.sin(theta) * t

### כלל 4: הוספת ציר עם None

בדוגמה ובבאג שלמעלה השתמשנו כבר ב-‎`v0_runs[:, None]`‎ בלי להסביר לעומק. הגיע הזמן לכלל הרביעי:

```{admonition} כלל 4 — None
:class: tip
‎`None`‎ בתוך פריסה מוסיף **ציר חדש בגודל 1** במיקום הזה — כלומר "המערך הזה קבוע (לא משתנה) לאורך הציר החדש".
```

‎`v0_runs[:, None]`‎ הופך ‎`(5,)`‎ ל-‎`(5, 1)`‎: "לכל אחת מ-5 הריצות, יש עמודה אחת (עדיין) — אבל אפשר להתאים אותה כנגד כל ציר נוסף".

אפשר לשים ‎`None`‎ גם בהתחלה: ‎`a[None, :]`‎ הופך את ‎`a`‎ לשורה. התא הבא מדפיס את שתי האפשרויות:

In [ ]:
a = np.array([1.0, 2.0])
print(a[:, None])    # a column: shape (2, 1)
print(a[None, :])    # a row: shape (1, 2)

### דוגמה: טבלת טווחים לכל מהירות ולכל זווית

לפעמים רוצים לחשב גודל פיזיקלי לכל צירוף של שני פרמטרים, למשל את הטווח $R=v_0^2\sin(2\theta)/g$ לכל מהירות שיגור **ולכל** זווית. מוסיפים ציר לכל אחד מהם: המהירויות הופכות לעמודה (‎`v0_vals[:, None]`‎), הזוויות לשורה (‎`theta_vals[None, :]`‎), ו-broadcasting בונה את הטבלה כולה.

In [ ]:
g = 10.0                                  # m/s^2, rounded for round numbers
v0_vals = np.array([10.0, 20.0])          # launch speeds (m/s)
angles_deg = np.array([15.0, 45.0, 75.0])
theta_vals = np.radians(angles_deg)

R = v0_vals[:, None]**2 * np.sin(2 * theta_vals[None, :]) / g
print(R)

בטבלה שורה לכל מהירות ועמודה לכל זווית. למשל, ‎`R[1, 2]`‎ הוא הטווח במהירות של 20 מטר לשנייה ובזווית ‎75°‎: ‏20 מטרים. בכל שורה הטווח הגדול ביותר נמצא בעמודה האמצעית, של ‎45°‎.

### נסו בעצמכם

זמן המעוף של זריקה הוא $T=2v_0\sin\theta/g$. השלימו את הפונקציה ‎`flight_time_table`‎, שמחזירה טבלה של זמני מעוף: שורה לכל מהירות ועמודה לכל זווית. השתמשו ב-‎`None`‎, בלי לולאה.

In [ ]:
def flight_time_table(v0_vals, angles_deg, g):
    """Flight time (s): a row per launch speed, a column per angle (deg)."""
    pass   # replace with your code

print(flight_time_table(np.array([10.0, 20.0]), np.array([30.0, 90.0]), 10.0))
# expected:
# [[1. 2.]
#  [2. 4.]]

`````{admonition} פתרון
:class: dropdown, tip
```python
def flight_time_table(v0_vals, angles_deg, g):
    """Flight time (s): a row per launch speed, a column per angle (deg)."""
    theta = np.radians(angles_deg)
    return 2 * v0_vals[:, None] * np.sin(theta[None, :]) / g

print(flight_time_table(np.array([10.0, 20.0]), np.array([30.0, 90.0]), 10.0))
```
בזווית 30° הסינוס הוא 0.5: מהירות של 10 מטר לשנייה נותנת שנייה אחת, ו-20 מטר לשנייה נותנות שתיים. בזריקה אנכית (90°) הזמנים כפולים.
`````

### בדקו את עצמכם

שלוש השאלות מתייחסות לקוד הבא:

```python
a = np.array([1.0, 2.0])
b = np.array([10.0, 20.0, 30.0])
c = a[:, None] * b
print(c[1])       # line A
print(c[:, 0])    # line B
print(a * b)      # line C
```

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "מהו המספר האחרון שתדפיס שורה A?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "60.0",
                "correct": true,
                "feedback": "נכון: השורה השנייה של c היא 2.0 כפול כל ערך של b: 20, 40 ו-60."
            },
            {
                "answer": "30.0",
                "correct": false,
                "feedback": "לא: 30.0 הוא המספר האחרון בשורה הראשונה, 1.0 כפול 30."
            },
            {
                "answer": "40.0",
                "correct": false,
                "feedback": "לא: 40.0 הוא המספר האמצעי בשורה השנייה."
            },
            {
                "answer": "20.0",
                "correct": false,
                "feedback": "לא: 20.0 הוא המספר הראשון בשורה השנייה."
            }
        ]
    },
    {
        "question": "מהו המספר הראשון שתדפיס שורה B?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "10.0",
                "correct": true,
                "feedback": "נכון: העמודה הראשונה היא כל ערך של a כפול 10: 10 ו-20."
            },
            {
                "answer": "1.0",
                "correct": false,
                "feedback": "לא: כל איבר של c הוא מכפלה של ערך מ-a בערך מ-b."
            },
            {
                "answer": "20.0",
                "correct": false,
                "feedback": "לא: 20.0 הוא המספר השני בעמודה הראשונה."
            },
            {
                "answer": "30.0",
                "correct": false,
                "feedback": "לא: 30.0 נמצא בעמודה האחרונה, לא בראשונה."
            }
        ]
    },
    {
        "question": "מה יקרה בשורה C?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "תתקבל שגיאה, כי האורכים 2 ו-3 שונים ואף אחד מהם אינו 1",
                "correct": true,
                "feedback": "נכון: זה בדיוק הבאג של לשכוח להוסיף ציר עם None."
            },
            {
                "answer": "תודפס שורה אחת של 3 מספרים",
                "correct": false,
                "feedback": "לא: כדי לכפול איבר-איבר צריך אורכים שווים, או ציר בגודל 1."
            },
            {
                "answer": "תודפס טבלה של 2 שורות ו-3 עמודות",
                "correct": false,
                "feedback": "לא: טבלה כזו מתקבלת רק עם None, כמו ב-c."
            },
            {
                "answer": "יודפסו 2 מספרים",
                "correct": false,
                "feedback": "לא: NumPy לא מקצר את המערך הארוך; הוא עוצר בשגיאה."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)

``````{admonition} תרגול נוסף
:class: dropdown, note
**לפני שמריצים:** מה תדפיס כל אחת משתי שורות ההדפסה? באיזו מהן יש שורה לכל ערך של ‎`b`‎?

```python
a = np.array([1, 2])
b = np.array([10, 20, 30])
print(a[:, None] + b)
print(a[None, :] + b[:, None])
```

`````{admonition} פתרון
:class: dropdown, tip
```text
[[11 21 31]
 [12 22 32]]
[[11 12]
 [21 22]
 [31 32]]
```
בשורה הראשונה ‎`a`‎ הוא העמודה, ולכן יש שורה לכל ערך של ‎`a`‎ ועמודה לכל ערך של ‎`b`‎. בשנייה ‎`b`‎ הוא העמודה, ולכן יש שורה לכל ערך של ‎`b`‎. אותם סכומים, בסידור הפוך.
`````
``````